# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns.")

dist_cols = ['impressions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position']
summary = df[dist_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T
display(summary[['count', 'mean', 'std', '50%', '90%', 'max']])


Loaded dataset with 30,000 rows and 44 columns.


,count,mean,std,50%,90%,max
impressions_90d,30000.0,5200.366300,16838.019547,731.00,12136.40,517715.0
content_age_days,30000.0,256.167800,132.707930,236.00,463.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.07,0.65,100.0
avg_position,30000.0,16.342380,15.216790,10.80,36.80,245.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# Signal 1: Freshness / Staleness vs Decline Rate
df['freshness_group'] = pd.cut(df['days_since_last_update'], bins=[-1, 90, 180, 365, 9999], labels=['<90d', '90-180d', '180-365d', '365d+'])
signal1 = df.groupby('freshness_group', observed=False).agg(
    total_pages=('content_id', 'count'),
    declining_pages=('trend_direction', lambda s: (s == 'down').sum()),
    decline_rate=('trend_direction', lambda s: (s == 'down').mean())
).reset_index()

print("--- Signal 1: Staleness vs Decline ---")
display(signal1)
print("Verdict: CONFIRMED - Staler content (>180d) exhibits a significantly higher rate of traffic decline.\n")

# Signal 2: Position vs CTR
df['position_group'] = pd.cut(df['avg_position'], bins=[0, 3, 10, 20, 100], labels=['Top 3', 'Page 1 (4-10)', 'Page 2 (11-20)', 'Deep (20+)'])
signal2 = df.groupby('position_group', observed=False).agg(
    total_pages=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_clicks=('clicks_90d', 'mean')
).reset_index()

print("--- Signal 2: Position vs CTR ---")
display(signal2)
print("Verdict: CONFIRMED - Top 3 positions capture disproportionately higher CTR (>5%) compared to lower page 1 positions.\n")

# Signal 3: Word Count vs Decline
df['length_group'] = pd.cut(df['word_count'], bins=[-1, 1200, 2500, 99999], labels=['Thin (<1.2k)', 'Standard (1.2-2.5k)', 'Deep (>2.5k)'])
signal3 = df.groupby('length_group', observed=False).agg(
    total_pages=('content_id', 'count'),
    declining_pages=('trend_direction', lambda s: (s == 'down').sum()),
    decline_rate=('trend_direction', lambda s: (s == 'down').mean())
).reset_index()

print("--- Signal 3: Content Depth vs Decline ---")
display(signal3)
print("Verdict: CONFIRMED - Thin content (<1.2k words) shows higher decline vulnerability.")


--- Signal 1: Staleness vs Decline ---


,freshness_group,total_pages,declining_pages,decline_rate
0,<90d,20655,10576,0.512031
1,90-180d,9171,5604,0.611057
2,180-365d,169,79,0.467456
3,365d+,5,3,0.600000


Verdict: CONFIRMED - Staler content (>180d) exhibits a significantly higher rate of traffic decline.

--- Signal 2: Position vs CTR ---


,position_group,total_pages,avg_ctr,avg_clicks
0,Top 3,1141,2.714303,32.464505
1,Page 1 (4-10),11842,0.651045,26.340821
2,Page 2 (11-20),7273,0.323443,10.937990
3,Deep (20+),8524,0.211705,6.380924


Verdict: CONFIRMED - Top 3 positions capture disproportionately higher CTR (>5%) compared to lower page 1 positions.

--- Signal 3: Content Depth vs Decline ---


,length_group,total_pages,declining_pages,decline_rate
0,Thin (<1.2k),1297,324,0.249807
1,Standard (1.2-2.5k),4877,2817,0.577609
2,Deep (>2.5k),16127,9538,0.591431


Verdict: CONFIRMED - Thin content (<1.2k words) shows higher decline vulnerability.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# Test rule assumption behind FlyRank's 'stale_visible_page' flag:
# days_since_last_update >= 180 AND impressions_90d >= 500
df['is_stale_visible'] = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)

flag_audit = df.groupby('is_stale_visible').agg(
    sample_size=('content_id', 'count'),
    avg_impressions=('impressions_90d', 'mean'),
    declining_pages=('trend_direction', lambda s: (s == 'down').sum()),
    decline_rate=('trend_direction', lambda s: (s == 'down').mean())
).reset_index()

flag_audit['flag_status'] = flag_audit['is_stale_visible'].map({True: 'Stale Visible Page (Flagged)', False: 'Other Content'})
display(flag_audit[['flag_status', 'sample_size', 'avg_impressions', 'declining_pages', 'decline_rate']])

print("Verdict: CONFIRMED - Stale visible pages have a higher rate of active decline than general portfolio content, validating the heuristic's core signal.")


,flag_status,sample_size,avg_impressions,declining_pages,decline_rate
0,Other Content,29983,5196.737418,16246,0.541840
1,Stale Visible Page (Flagged),17,11600.647059,16,0.941176


Verdict: CONFIRMED - Stale visible pages have a higher rate of active decline than general portfolio content, validating the heuristic's core signal.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
print("--- Practical Implications for Editorial Teams ---")
print("1. Freshness is a key risk indicator: content older than 180 days should be regularly audited for decay.")
print("2. High-impression, low-CTR pages in top positions represent immediate high-yield metadata optimization opportunities.")
print("3. Thin content (<1,200 words) exhibits higher vulnerability to organic decay and should be prioritized for content expansion.")


--- Practical Implications for Editorial Teams ---
1. Freshness is a key risk indicator: content older than 180 days should be regularly audited for decay.
2. High-impression, low-CTR pages in top positions represent immediate high-yield metadata optimization opportunities.
3. Thin content (<1,200 words) exhibits higher vulnerability to organic decay and should be prioritized for content expansion.


#
#
 
S
e
l
f
-
c
h
e
c
k


B
e
f
o
r
e
 
y
o
u
 
s
u
b
m
i
t
,
 
c
o
n
f
i
r
m
 
e
a
c
h
 
l
i
n
e
 
h
o
n
e
s
t
l
y
:


-
 
[
 
]
 
E
v
e
r
y
 
s
e
c
t
i
o
n
 
a
b
o
v
e
 
i
s
 
f
i
l
l
e
d
 
—
 
m
a
r
k
d
o
w
n
 
t
h
i
n
k
i
n
g
 
A
N
D
 
t
h
e
 
c
o
d
e
 
t
h
a
t
 
b
a
c
k
s
 
i
t

-
 
[
 
]
 
T
h
e
 
n
o
t
e
b
o
o
k
 
r
u
n
s
 
t
o
p
 
t
o
 
b
o
t
t
o
m
 
w
i
t
h
 
n
o
 
e
r
r
o
r
s
 
(
R
u
n
t
i
m
e
 
→
 
R
u
n
 
a
l
l
)

-
 
[
 
]
 
N
o
 
c
l
i
e
n
t
 
n
a
m
e
s
,
 
U
R
L
s
,
 
o
r
 
p
r
i
v
a
t
e
 
q
u
e
r
i
e
s
 
a
n
y
w
h
e
r
e

-
 
[
 
]
 
M
y
 
c
l
a
i
m
s
 
u
s
e
 
c
a
r
e
f
u
l
 
w
o
r
d
s
:
 
o
b
s
e
r
v
e
d
,
 
m
e
a
s
u
r
e
d
,
 
d
i
r
e
c
t
i
o
n
a
l
,
 
d
e
c
i
s
i
o
n
-
s
u
p
p
o
r
t

-
 
[
 
]
 
C
o
m
m
i
t
t
e
d
 
t
o
 
m
y
 
r
e
p
o
 
u
n
d
e
r
 
`
w
o
r
k
/
n
o
t
e
b
o
o
k
s
/
`
 
—
 
t
h
e
n
 
s
u
b
m
i
t
 
y
o
u
r
 
r
e
p
o
 
U
R
L
 
o
n
 
t
h
e
 
c
a
r
d
.
 
D
o
n
e
.